Building with the Claude API: How to use tool calling

In [24]:
#Load env variables and create an Anthropic client
import json
from datetime import datetime
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [25]:
messages = [{ "role": "user", "content": "What is the current time" }]

response = client.messages.create(
    model=model,
    max_tokens=1024,
    messages=messages
)

print(response.content[0].text)

I don't have access to real-time information, so I can't tell you the current time. 

To find out the current time, you can:
- Check your device (phone, computer, watch)
- Search "current time" in your web browser
- Ask your voice assistant (Siri, Google Assistant, Alexa, etc.)

Is there anything else I can help you with?


In [26]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    return datetime.now().strftime(date_format)

curent_time = get_current_datetime("%Y-%m-%d %H:%M:%S")
print(curent_time)

2026-10-02 00:00:52


In [27]:
get_current_datetime_schema = {
        "name": "get_current_datetime",
        "description": "Returns the current date and time formatted according to the specified format string. This tool provides the current system time formatted as a string. Use this tool when you need to know the current date and time, such as for timestamping records, calculating time differences, or displaying the current time to users. The default format returns the date and time in ISO-like format (YYYY-MM-DD HH:MM:SS).",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_format": {
                    "type": "string",
                    "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes. For example, '%Y-%m-%d' returns just the date in YYYY-MM-DD format, '%H:%M:%S' returns just the time in HH:MM:SS format, '%B %d, %Y' returns a date like 'May 07, 2025'. The default is '%Y-%m-%d %H:%M:%S' which returns a complete timestamp like '2025-05-07 14:32:15'.",
                    "default": "%Y-%m-%d %H:%M:%S",
                }
            },
            "required": [],
        },
    }

In [28]:

messages = [{ "role": "user", "content": "What is the current time" }]
tools = [get_current_datetime_schema]

while True:
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        tools=tools,
        messages=messages,
    )

    if response.stop_reason != "tool_use":
        break

    messages.append({"role": "assistant", "content": response.content})

    results = []
    for block in response.content:
        if block.type == "tool_use":
            try:
                output = get_current_datetime(**block.input)
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(output),
                })
            except Exception as e:
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(e),
                    "is_error": True,
                })

    messages.append({"role": "user", "content": results})

print(response.content[0].text)

The current time is **00:01:00** (just after midnight) on **October 2, 2026**.
